### Stock Forecasting with LSTM & GRU Models (2020–2024)

This notebook implements end-to-end stock price forecasting using both **LSTM** and **GRU** architectures for Israeli tickers from 2020 to 2024. Historical data is fetched using `yfinance`, enriched with engineered features (rolling averages, RSI, MACD, volatility, and return trends), and used to train models with a **volatility-sensitive loss** to capture both level and variability in stock movements.

Both LSTM and GRU models share the same deep architecture:
- **4 stacked recurrent layers** with 100 units each and dropout regularization,
- Followed by **batch normalization** and **2 dense layers**, ending with a **single output neuron** predicting the adjusted close price.

After training:
- The model forecasts are generated by computing **daily percent change statistics** (mean and std) from test predictions.
- It **samples new daily returns** from a normal distribution and applies them iteratively over 30 future days, starting from the last predicted price.

For each stock, the pipeline produces:
- Actual vs Predicted values,
- 30-day price forecasts,
- Evaluation metrics (MSE, MAE, MAPE, R²),
- **LIME-based interpretability reports** showing feature impact.

All outputs are saved as CSV files, ready for downstream analysis or integration into a financial assistant app.


In [ ]:
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=07959f8a6273f6c8bcaae407fbc67fba276287e71681bca4909b3f59d88424c9
  Stored in directory: /root/.cache/pip/wheels/85/fa/a3/9c2d44c9f3cd77cf4e533b58900b2bf4487f2a17e8ec212a3d
Successfully built lime


In [ ]:
!pip install shap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## LSTM model:

In [ ]:
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, BatchNormalization
import tensorflow.keras.backend as K
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error
from google.colab import drive
import yfinance as yf
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import tensorflow as tf
from scipy.optimize import minimize
from lime.lime_tabular import LimeTabularExplainer
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
def LSTM_model(tickers, start_date, end_date):
  # Initialize a list for failed downloads
  failed_tickers = []

  # Download data while skipping missing tickers
  valid_tickers = []
  data_frames = []

  # metrics_lists:
  mse_list = []
  mae_list = []
  mape_list = []
  r2_list = []

  # Initialize DataFrames to store results
  actual_vs_pred_df = pd.DataFrame(columns=['Date', 'Ticker', 'Actual', 'Predicted'])
  forecast_df = pd.DataFrame(columns=['Date', 'Ticker', 'Forecast', 'Days_Ahead'])
  metrics_df = pd.DataFrame(columns=['Ticker', 'MSE', 'MAE','MAPE', 'R2'])
  for ticker in tickers:
    try:
        print(f"Downloading data for: {ticker}")
        data = yf.download(ticker, start_date, end_date, progress=False)
        if not data.empty:
            data.columns = ['Close', 'High', 'Low', 'Open', 'Volume']
            data['Adj Close']=data['Close']
            data['Ticker'] = ticker  # Add a column to identify the ticker
            data_frames.append(data)
            valid_tickers.append(ticker)
        else:
            failed_tickers.append(ticker)
    except Exception as e:
        print(f"Failed to download data for {ticker}: {e}")
        failed_tickers.append(ticker)

  # Concatenate data for all valid tickers
  if data_frames:
    combined_data = pd.concat(data_frames)
  else:
    print("No valid tickers downloaded.")

  # Print tickers that failed
  if failed_tickers:
    print("\nThe following tickers could not be downloaded:")
    print(failed_tickers)
  else:
    print("\nAll tickers were successfully downloaded.")

  # Flatten the multi-level columns
  data=combined_data.copy()
  # Create custom features: High-Low difference and Open-Close difference
  grouped_data = {}

  # RSI Calculation
  def calculate_rsi(prices, period=14):
    prices = pd.Series(prices).astype(float).dropna()
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

  # MACD Calculation
  def calculate_macd(prices, fast=12, slow=26):
    prices = pd.Series(prices).astype(float).dropna()
    exp1 = prices.ewm(span=fast, adjust=False).mean()
    exp2 = prices.ewm(span=slow, adjust=False).mean()
    return exp1 - exp2
  def model_predict(inputs, model, scaler_y):
    predictions = model.predict(inputs)
    return scaler_y.inverse_transform(predictions).flatten()

  # Function to explain predictions with LIME
  def explain_with_lime(ticker, model, X_scaled, y_test_inverse, scaler_X, scaler_y, features):
      print(f"Explaining predictions for ticker: {ticker}")

      # Initialize LIME explainer
      explainer = LimeTabularExplainer(
          training_data=scaler_X.inverse_transform(X_scaled.reshape(X_scaled.shape[0], X_scaled.shape[2])),
          mode="regression",
          feature_names=features,
          verbose=True,
          random_state=42
      )

      # Select a random test sample for explanation
      sample_index = np.random.randint(0, len(y_test_inverse))
      sample = X_scaled[sample_index].reshape(1, 1, -1)

      # Generate explanation
      explanation = explainer.explain_instance(
          data_row=scaler_X.inverse_transform(sample.reshape(-1, sample.shape[2])).flatten(),
          predict_fn=lambda x: model_predict(x.reshape(x.shape[0], 1, x.shape[1]), model, scaler_y),
          num_features=10  # Number of top features to display
      )

      # Display the explanation
      explanation.show_in_notebook(show_table=True)
      explanation.save_to_file(f'lime_explanation_{ticker}_lstm.html')

      print(f"LIME explanation saved for ticker: {ticker}")

  # Perform calculations for each ticker
  for ticker in tickers:
    # Extract data for the specific ticker
    ticker_data = data[data['Ticker']==ticker]

    # # Calculate High-Low and Open-Close differences
    ticker_data['High_Low_Diff'] = ticker_data['High'] - ticker_data['Low']
    ticker_data['Open_Close_Diff'] = ticker_data['Open'] - ticker_data['Close']

    # Add Rolling Averages
    ticker_data['Adj_Close_5d_Rolling'] = ticker_data['Adj Close'].rolling(window=5).mean()
    ticker_data['Adj_Close_10d_Rolling'] = ticker_data['Adj Close'].rolling(window=10).mean()
    ticker_data['Adj_Close_20d_Rolling'] = ticker_data['Adj Close'].rolling(window=20).mean()
    ticker_data['Volatility'] = ticker_data['Adj Close'].rolling(window=10).std()
    # Add Binary Feature: Is Adj Close up (1) or down (0) compared to previous day
    ticker_data['Adj_Close_Up'] = (ticker_data['Adj Close'].diff() > 0).astype(int)
    ticker_data['RSI'] = calculate_rsi(ticker_data['Adj Close'])
    ticker_data['MACD'] = calculate_macd(ticker_data['Adj Close'])
    ticker_data['Returns'] = ticker_data['Adj Close'].pct_change()

    # Keep only the relevant columns
    grouped_data[ticker] = ticker_data[['Adj Close', 'Volume',
                                        'High_Low_Diff', 'Open_Close_Diff', 'High', 'Low', 'Open', 'Close',
                                      'Adj_Close_5d_Rolling',
                                      'Adj_Close_10d_Rolling',
                                      'Adj_Close_20d_Rolling', 'Volatility', 'Adj_Close_Up', 'RSI', 'MACD', 'Returns']]

  # Combine all tickers into a MultiIndex DataFrame
  data_filtered = pd.concat(grouped_data, axis=1)

  if '2024-04-07' in data_filtered.index:
    # Get the index position of '2024-04-07'
    idx = data_filtered.index.get_loc('2024-04-07')

    # Replace each column's value for '2024-04-07' with its next row's value
    for col in data_filtered.columns:
      if idx + 1 < len(data_filtered):  # Ensure the next row exists
          next_value = data_filtered.iloc[idx + 1][col]
          data_filtered.loc['2024-04-07', col] = next_value
  # Input parameters
  tickers = valid_tickers  # Add all tickers here
  num_days_to_predict = 30  # Forecast days



  def volatility_loss(y_true, y_pred):
    # Compute the difference between consecutive elements for volatility
    diff_true = K.abs(y_true[:, 1:] - y_true[:, :-1])
    diff_pred = K.abs(y_pred[:, 1:] - y_pred[:, :-1])

    # Compute the volatility weight
    volatility_weight = K.clip(diff_true, 1, 10)  # Limit weight to avoid over-amplification

    # Align shapes for loss computation
    mse_loss = K.mean(K.square(y_true - y_pred))  # Mean squared error for the main prediction
    volatility_loss = K.mean(volatility_weight * K.square(diff_true - diff_pred))  # Penalize volatility mismatch

    return mse_loss + 0.1 * volatility_loss  # Combine both losses


  # Function to create LSTM model
  def create_lstm_model(input_shape):
    model = Sequential()
    model.add(Input(shape=input_shape))
    model.add(LSTM(100, return_sequences=True))
    model.add(Dropout(0.1))
    model.add(LSTM(100, return_sequences=True))
    model.add(Dropout(0.1))
    model.add(LSTM(100, return_sequences=True))
    model.add(Dropout(0.1))
    model.add(LSTM(100, return_sequences=False))
    model.add(BatchNormalization())
    model.add(Dropout(0.1))
    model.add(Dense(25))
    model.add(Dropout(0.1))
    model.add(Dense(25))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss=volatility_loss)
    return model

  # Function to train and forecast for each stock
  def process_stock(ticker, data_filtered, actual_vs_pred_df, forecast_df, metrics_df):

    print(f"Processing ticker: {ticker}")

    # Extract data for the ticker
    ticker_data = data_filtered.xs(ticker, axis=1, level=0)
    ticker_data = ticker_data.dropna()

    # Features and target
    X = ticker_data.drop('Adj Close', axis=1)
    y = ticker_data['Adj Close']
    dates = ticker_data.index

    # Scaling
    scaler_X, scaler_y = RobustScaler(), RobustScaler()
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1))
    X_scaled = X_scaled.reshape(X_scaled.shape[0], 1, X_scaled.shape[1])

    # Split data
    split_index = int(0.9 * len(X_scaled))
    X_train, X_test = X_scaled[:split_index], X_scaled[split_index:]
    y_train, y_test = y_scaled[:split_index], y_scaled[split_index:]
    train_dates, test_dates = dates[:split_index], dates[split_index:]

    # Build LSTM model
    model = create_lstm_model(input_shape=(X_train.shape[1], X_train.shape[2]))
    model.fit(X_train, y_train, epochs=30, batch_size=32, verbose=0)

    # Define model path
    model_path = f"/content/drive/Shareddrives/capstone project-stock market robo-advisor/LSTM models/{ticker.replace('.', '_')}_lstm_model.h5"

    # Save the model
    model.save(model_path)
    print(f"✅ LSTM model for {ticker} saved to: {model_path}")

    # Predict
    y_pred = model.predict(X_test)


    y_pred_inverse = scaler_y.inverse_transform(y_pred)
    y_test_inverse = scaler_y.inverse_transform(y_test)
    if np.any(np.isnan(y_test_inverse)) or np.any(np.isnan(y_pred_inverse)):
        print(f"NaN values detected for ticker: {ticker}")
        # Remove NaN values
        y_test_inverse = y_test_inverse[~np.isnan(y_test_inverse)]
        y_pred_inverse = y_pred_inverse[~np.isnan(y_pred_inverse)]

    # Ensure both arrays have the same length
    if len(y_test_inverse) == 0 or len(y_pred_inverse) == 0:
        print(f"No valid predictions for ticker: {ticker}. Skipping metric calculation.")
        return

    if len(y_test_inverse) != len(y_pred_inverse):
        min_length = min(len(y_test_inverse), len(y_pred_inverse))
        y_test_inverse = y_test_inverse[:min_length]
        y_pred_inverse = y_pred_inverse[:min_length]
    # Calculate metrics
    mse = mean_squared_error(y_test_inverse, y_pred_inverse)
    mae = mean_absolute_error(y_test_inverse, y_pred_inverse)
    mape = mean_absolute_percentage_error(y_test_inverse, y_pred_inverse)
    r2 = r2_score(y_test_inverse, y_pred_inverse)

    print(f"Ticker: {ticker}, MSE: {mse:.3f}, MAE: {mae:.3f}, MAPE: {mape:.3f}, R2: {r2:.3f}")
    # append metrics to metrics' lists:
    mse_list.append(mse)
    mae_list.append(mae)
    mape_list.append(mape)
    r2_list.append(r2)

    # Append metrics to DataFrame
    metrics_df.loc[len(metrics_df)] = [ticker, mse, mae, mape, r2]

    # Append actual vs predicted values
    temp_actual_pred = pd.DataFrame({
        'Date': test_dates,
        'Ticker': ticker,
        'Actual': y_test_inverse.flatten(),
        'Predicted': y_pred_inverse.flatten()
    })
    temp_actual_pred = temp_actual_pred[['Date', 'Ticker', 'Actual', 'Predicted']]  # Ensure correct column order
    actual_vs_pred_df = pd.concat([actual_vs_pred_df, temp_actual_pred], ignore_index=True)


    # Forecast next n days
    future_predictions = [y_pred_inverse[-1]]
    for day in range(1, num_days_to_predict + 1):
        # Compute percent changes from test predictions
        pct_changes = pd.Series(y_pred_inverse.flatten()).pct_change().dropna()
        mean_pct_change = pct_changes.mean()
        std_pct_change = pct_changes.std()
        # Start with the last predicted value
        sampled_pct_change = np.random.normal(loc=mean_pct_change, scale=std_pct_change)
        new_forecast = future_predictions[-1] * (1 + sampled_pct_change)
        future_predictions.append(new_forecast)
        y_pred_inverse.append(new_forecast)
    y_pred_inverse=y_pred_inverse[:-num_days_to_predict]
    # Convert to DataFrame
    future_dates = pd.date_range(dates[-1] + pd.Timedelta(days=1), periods=num_days_to_predict)
    temp_forecast = pd.DataFrame({
        'Date': future_dates,
        'Ticker': ticker,
        'Forecast': future_predictions[1:],  # Exclude initial value
        'Days_Ahead': np.arange(1, num_days_to_predict + 1)
    })



    # Append forecasted values
    temp_forecast = temp_forecast[['Date', 'Ticker', 'Forecast', 'Days_Ahead']]  # Ensure correct column order
    forecast_df = pd.concat([forecast_df, temp_forecast], ignore_index=True)




    # Plot results
    plt.figure(figsize=(14, 8))
    plt.plot(train_dates, scaler_y.inverse_transform(y_train), color='green', label='Actual Adj Close (Train)')
    plt.plot(test_dates, y_test_inverse, color='blue', label='Actual Adj Close (Test)')
    plt.plot(test_dates, y_pred_inverse, color='red', label='Predicted Adj Close (Test)')
    plt.plot(temp_forecast['Date'], temp_forecast['Forecast'], color='orange', label='Forecasted Prices')

    plt.title('Stock Prices: Actual, Predicted, and Forecasted')
    plt.xlabel('Date')
    plt.ylabel('Adj Close')
    plt.legend()
    plt.show()

    feature_names = X.columns  # Update to match your feature names
    explain_with_lime(ticker, model, X_scaled, y_test_inverse, scaler_X, scaler_y, feature_names)
    return actual_vs_pred_df, forecast_df, metrics_df

  for ticker in tickers:
    actual_vs_pred_df, forecast_df, metrics_df = process_stock(ticker, data_filtered, actual_vs_pred_df, forecast_df, metrics_df)
  return actual_vs_pred_df, forecast_df, metrics_df, mse_list, mae_list, mape_list, r2_list
df_tickers = pd.read_csv("/content/drive/Shareddrives/capstone project-stock market robo-advisor/symbols_Israel.csv")
tickers = df_tickers["0"].tolist()

start_date = "2020-01-01"
end_date = "2024-12-31"
actual_vs_pred_df, forecast_df, metrics_df, mse_list, mae_list, mape_list, r2_list=LSTM_model(tickers, start_date, end_date)
actual_vs_pred_df.to_csv('actual_vs_pred_lstm.csv', index=False)
forecast_df.to_csv('forecast_lstm.csv', index=False)
metrics_df.to_csv('metrics_lstm.csv', index=False)

In [2]:
arr = [1, 2, 3, 4, 5]
n = 2
arr = arr[:-n]  # Removes last n elements
print(arr)  # Output: [1, 2, 3]


[1, 2, 3]
